Paper: https://aclanthology.org/2021.emnlp-main.612.pdf

Github: https://github.com/nattaptiy/qe_disentangled

Eval data of paper: 
- Task 3 Document-Level QA: https://github.com/facebookresearch/mlqe
- STS17: https://public.ukp.informatik.tu-darmstadt.de/reimers/sentence-transformers/datasets/STS2017-extended.zip

In [22]:
import torch
import torch.nn as nn

class LanguageEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super(LanguageEmbedding, self).__init__()
        self.embedding = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, sentence_embeddings):
        return self.embedding(sentence_embeddings)
    
class MeaningEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super(MeaningEmbedding, self).__init__()
        self.embedding = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, sentence_embeddings):
        return self.embedding(sentence_embeddings)
    
class LanguageIdentifier(nn.Module):
    def __init__(self, embedding_dim, num_languages=8):
        super(LanguageIdentifier, self).__init__()
        self.classifier = nn.Linear(embedding_dim, num_languages)

    def forward(self, language_embedding):
        return self.classifier(language_embedding)
    
class DREAMModel(nn.Module):
    def __init__(self, embedding_dim):
        super(DREAMModel, self).__init__()
        self.language_embedding = LanguageEmbedding(embedding_dim)
        self.meaning_embedding = MeaningEmbedding(embedding_dim)
        self.language_identifier = LanguageIdentifier(embedding_dim)

    def forward(self, sentence_embeddings):
        language_embedded = self.language_embedding(sentence_embeddings)
        meaning_embedded = self.meaning_embedding(sentence_embeddings)
        language_id = self.language_identifier(language_embedded)
        return language_embedded, meaning_embedded, language_id

In [23]:
model = DREAMModel(embedding_dim=768)

# Kiểm tra structure
print(model)
# DREAMModel(
#   (language_embedding): LanguageEmbedding(
#     (embedding): Linear(in_features=768, out_features=768, bias=True)
#   )
#   (meaning_embedding): MeaningEmbedding(...)
#   (language_identifier): LanguageIdentifier(
#     (classifier): Linear(in_features=768, out_features=8, bias=True)
#   )
# )

# Kiểm tra forward pass
e = torch.randn(512, 768)   # batch 512, mBERT embedding
lang_emb, mean_emb, lang_id = model(e)

assert lang_emb.shape == (512, 768)
assert mean_emb.shape == (512, 768)
assert lang_id.shape  == (512, 8)
print("✅ All shapes correct")

# Kiểm tra gradient flow đúng chỗ
loss = lang_id.sum()
loss.backward()

# lang_id ← language_identifier ← language_embedding
# → gradient phải chảy vào language_embedding
assert model.language_embedding.embedding.weight.grad is not None   # ✅

# → gradient KHÔNG chảy vào meaning_embedding từ lang_id
assert model.meaning_embedding.embedding.weight.grad is None        # ✅
print("✅ Gradient flow correct")

DREAMModel(
  (language_embedding): LanguageEmbedding(
    (embedding): Linear(in_features=768, out_features=768, bias=True)
  )
  (meaning_embedding): MeaningEmbedding(
    (embedding): Linear(in_features=768, out_features=768, bias=True)
  )
  (language_identifier): LanguageIdentifier(
    (classifier): Linear(in_features=768, out_features=8, bias=True)
  )
)
✅ All shapes correct
✅ Gradient flow correct


# WMT20 QE Dataset
Languages:
- English-German (en-de)
- English-Chinese (en-zh)
- Romanian-English (ro-en)
- Estonian-English (et-en)
- Nepalese-English (ne-en)
- Sinhala-English (si-en)

Size: 5% of parallel sentence pairs per language pair